## Requirements

In [ ]:
!pip install langchain-community pypdf pypdfium2 langchain-openai chromadb faiss-cpu -qq
!pip install huggingface_hub -qq
!pip install langchain-groq -qq
!pip install -U langchain langchain-community sentence-transformers langchain-google-genai -qq


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.9/67.9 kB 2.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 48.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.6/330.6 kB 25.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 75.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.8/85.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.4/21.4 MB 56.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 52.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 51.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 500.5/500.5 kB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1

## Important Libraries

In [ ]:
import os
from google.colab import userdata
from langchain_community.document_loaders import PyPDFLoader, PyPDFium2Loader
from langchain_community.vectorstores import Chroma , FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings

os.environ["HUGGINGFACEHUB_API_TOKEN"] = userdata.get("huggingface")
os.environ["GROQ_API_KEY"] = userdata.get("groq")


## DataLoader

In [ ]:
from langchain_community.document_loaders import TextLoader
arabic_loader = TextLoader("/content/arabic.txt", encoding="utf-8")
arabic_documents = arabic_loader.load()

print(arabic_documents[0])

page_content='--- START OF FILE ai_studio_code (2).txt ---

تاريخ القلعة البيضاء: من الحصن المنيع إلى منارة الثقافة (النسخة الموسعة والشاملة)

تضرب جذور "القلعة البيضاء" في أعماق التاريخ المعماري والعسكري، حيث تتوج قمة الهضبة الجيرية التي تشرف باستعلاء على المدينة القديمة ومسارات القوافل التجارية القديمة. يعود تأسيس هذا الصرح المهيب إلى القرن الرابع عشر الميلادي، وتحديداً في حقبة المماليك، حين أصدر السلطان "المنصور قلاوون" مرسوماً سلطانياً بتشييد منظومة دفاعية استراتيجية لتكون درعاً واقياً ضد الغزوات الخارجية، ومركزاً سياسياً لإدارة دفة الحكم. استغرقت عملية البناء المعقدة ما يربو على العشرين عاماً، سُخرت فيها إمكانيات الدولة المادية والبشرية، حيث توافد إليها نخبة من المهندسين والمعماريين والحرفيين من أصقاع الأندلس وبلاد الشام ومصر. وقد أثمر هذا التلاقح الثقافي عن تحفة معمارية نادرة تجسد "الطراز المملوكي المتأخر"، جامعةً بين صرامة التحصينات العسكرية—المتمثلة في الأسوار السميكة ذات المزاغل الدفاعية والأبراج الأسطوانية الشاهقة—وبين رقة الفنون الإسلامية التي تجلت في المشربيات الخشبية المعش

In [ ]:
print("Loaded Arabic document content (first document):")
if arabic_documents:
    print(arabic_documents[0].page_content)
else:
    print("No Arabic documents loaded.")

## Chunking

In [ ]:

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=550,
    chunk_overlap=50
    )

docs = text_splitter.split_documents(arabic_documents)

print(docs[0].page_content)


--- START OF FILE ai_studio_code (2).txt ---

تاريخ القلعة البيضاء: من الحصن المنيع إلى منارة الثقافة (النسخة الموسعة والشاملة)
9


In [ ]:
# Number of Chunks
print(f" Number of chunks is : {len(docs)}")

 Number of chunks is : 9 chunks


## Embedding

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer('BAAI/bge-m3')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

In [ ]:
texts = [doc.page_content for doc in docs]
docs_embed = model.encode(texts)

print(len(docs_embed[0]))

Another Embedding Model

In [ ]:

# embeddings = HuggingFaceEmbeddings(
#     model_name="sentence-transformers/all-MiniLM-L6-v2"
# )

# texts = [doc.page_content for doc in splits]
# docs_embed = embeddings.embed_documents(texts)

# print(len(docs_embed), len(docs_embed[0]))


## Store Embeddings in Vector Database

In [ ]:
embeddings_model = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")

Vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings_model,
   # collection_name="rag_collection"
)

#question="what is LLM?"
#result= Vectorstore.similarity_search(question , k=1)
#print({result[0].page_content})


/tmp/ipython-input-3661994334.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings_model = HuggingFaceEmbeddings(model_name="BAAI/bge-m3")


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

## Retriever

In [ ]:

retriever = Vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={
        "top_k": 3,
       "fetch_k": 10,
        "k": 2,
        "lambda_mult":0.75
        }
)




In [ ]:
q="ما هو تاريخ القلعه البيضاء?"
result = retriever.invoke(q)
print(result[0].page_content)

تضرب جذور "القلعة البيضاء" في أعماق التاريخ المعماري والعسكري، حيث تتوج قمة الهضبة الجيرية التي تشرف باستعلاء على المدينة القديمة ومسارات القوافل التجارية القديمة. يعود تأسيس هذا الصرح المهيب إلى القرن الرابع عشر الميلادي، وتحديداً في حقبة المماليك،


## Initialize LLM

In [ ]:
from google.colab import userdata
from langchain_google_genai import GoogleGenerativeAI

GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY")

llm = GoogleGenerativeAI(model="models/gemini-2.5-flash", api_key=GOOGLE_API_KEY)

## Prompt & Parser

In [ ]:

from langchain_core.prompts import PromptTemplate

prompt= PromptTemplate(
    template="""
         You are an AI assistant. Use the following context to answer the question.
         If the answer is not present in the context, say that you don't know.
         ,Ensuring that you respond with the same language as the question.
         provide the context you retrieved from the document with your answer.
         {context}
        context: {context},
        question: {question}

    """,
    input_variables=["context", "question"]
    )




from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()






## Build RAG Chain

In [ ]:
# read all docs after splitting
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


rag_chain = (
    {"context": retriever |format_docs,
     "question": lambda x: x}
    | prompt
    | llm
    | parser
)

## Test

In [ ]:
user_question = "ما هو تاريخ القلعة الحمراء"
response = rag_chain.invoke(user_question)
print(response)

لا أعرف.


## Simple UI Using Gradio To Chat with RAG Chatbot

In [ ]:
import gradio as gr

In [ ]:
def chat_function(message, history):

    response = rag_chain.invoke(message)

    return response

In [ ]:
iface = gr.ChatInterface(
    chat_function,
    title="RAG Chatbot with Document Context",
    description="Ask questions in context of  'Q&A AI Interview.pdf' document."
)

iface.launch(share=True)

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://cc0af0295c819d50e6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
